# 实验七：大语言模型实验

本 Notebook 对应 `exp7_llm_rag_agent.py`，包含三部分：提示工程、RAG 企业政策问答、简易 Agent 模拟。

默认使用 `mock` 后端，无 API Key 也能先跑通；正式实验时把 `backend` 改成 `zhipuai`、`deepseek` 或 `openai`，并在 `.env` 中配置 API Key。

本实验不训练本地大模型，只做 API 调用、文本向量化和 FAISS CPU 检索，因此普通 PC 即可运行。


In [1]:
# 依赖安装示例（需要时取消注释执行）
# %pip install -U openai zhipuai python-dotenv pypdf faiss-cpu numpy pandas

from pathlib import Path
from IPython.display import Markdown, display


In [2]:
from exp7_llm_rag_agent import (
    ExperimentConfig,
    ChatClient,
    EmbeddingClient,
    run_prompt_engineering,
    run_rag_experiment,
    run_agent_experiment,
    set_seed,
)

set_seed(42)

# mock：离线演示；zhipuai：智谱AI；deepseek：DeepSeek 聊天生成；openai：OpenAI 或兼容接口；auto：自动检测 API Key。
backend = 'deepseek'

cfg = ExperimentConfig(
    backend=backend,
    output_dir=Path('outputs_exp7'),
    kb_dir=Path('knowledge_base_exp7'),
    rebuild_kb=True,
    top_k=4,
)

chat_client = ChatClient(backend=cfg.backend, temperature=cfg.temperature, max_tokens=cfg.max_tokens)
embedding_client = EmbeddingClient(backend=cfg.backend, dimensions=cfg.embedding_dim)

print('LLM 后端：', chat_client.backend, chat_client.model)
print('Embedding 后端：', embedding_client.backend, embedding_client.model)


DeepSeek 后端用于聊天生成；Embedding 使用离线 mock-hash 兜底。
LLM 后端： deepseek deepseek-v4-flash
Embedding 后端： mock mock-hash-embedding


## 第一部分：提示工程实践


In [3]:
prompt_result = run_prompt_engineering(chat_client, cfg.output_dir)
display(Markdown(Path(prompt_result['markdown_path']).read_text(encoding='utf-8')))



========== 第一部分：提示工程实践 ==========


提示工程结果已保存：outputs_exp7\exp7_prompt_engineering_results.md


# 实验七第一部分：提示工程实践结果

生成时间：2026-06-04 11:14:26
LLM 后端：deepseek / deepseek-v4-flash

## 简要分析（100字以内）
CRISPE 提示词明确了背景、角色、输入、步骤、限制和输出格式，能让模型按要求逐项检查；糟糕提示词缺少角色、评分维度和格式约束，输出容易泛泛而谈，难以用于报告对比。

## 作文：难忘的运动会（essay_1）

### 原文
今天学校举行了一年一度的运动会。早晨操场上人山人海，同学们都非常兴奋。我报名参加了八百米比赛，刚开始我跑得很快，可是到了第二圈就觉得胸口很闷，脚步也慢了下来。班主任在跑道边不停给我鼓厉，同学们也喊着我的名字。听到大家的加油，我仿佛又有了力气，终于坚持到了终点。虽然我没有取得第一名，但是我明白了坚持比名次更重要。这次运动会让我受益非浅，也让我更加热爱班集体。

### 完整 CRISPE 提示词
```text
你将使用 CRISPE 框架完成作文批改任务。

C - Context（背景）：这是一篇初中生短文，老师需要你做基础语文批改，重点关注错别字、病句、评分与总体评价。
R - Role（角色）：你是认真、客观、表达温和的语文作文批改助手。
I - Input（输入）：作文题目和正文如下。
题目：难忘的运动会
正文：今天学校举行了一年一度的运动会。早晨操场上人山人海，同学们都非常兴奋。我报名参加了八百米比赛，刚开始我跑得很快，可是到了第二圈就觉得胸口很闷，脚步也慢了下来。班主任在跑道边不停给我鼓厉，同学们也喊着我的名字。听到大家的加油，我仿佛又有了力气，终于坚持到了终点。虽然我没有取得第一名，但是我明白了坚持比名次更重要。这次运动会让我受益非浅，也让我更加热爱班集体。
S - Steps（步骤）：
1. 通读全文，只依据作文原文进行判断，不要虚构不存在的问题。
2. 标注作文中的错别字，给出“原词/正确写法/说明”；如果没有错别字，写“未发现明显错别字”。
3. 找出至少 1 处病句或表达不顺的句子，给出修改建议。
4. 从内容、结构、语言三个维度打分，每项满分 10 分，并给出一句理由。
5. 给出 50 字以内的总体评价，语气鼓励但要指出改进方向。
P - Parameters（限制）：回答必须简洁清楚；不要超过 500 字；不要输出与作文无关的内容。
E - Evaluation（输出格式）：请严格按下面 Markdown 格式输出：

## 1. 错别字
| 原词 | 正确写法 | 说明 |
|---|---|---|

## 2. 病句与修改
| 原句 | 问题 | 修改建议 |
|---|---|---|

## 3. 三维评分
| 维度 | 分数/10 | 理由 |
|---|---:|---|
| 内容 |  |  |
| 结构 |  |  |
| 语言 |  |  |

## 4. 总体评价（50字以内）
```

### 糟糕提示词
```text
帮我看看这篇作文写得怎么样：今天学校举行了一年一度的运动会。早晨操场上人山人海，同学们都非常兴奋。我报名参加了八百米比赛，刚开始我跑得很快，可是到了第二圈就觉得胸口很闷，脚步也慢了下来。班主任在跑道边不停给我鼓厉，同学们也喊着我的名字。听到大家的加油，我仿佛又有了力气，终于坚持到了终点。虽然我没有取得第一名，但是我明白了坚持比名次更重要。这次运动会让我受益非浅，也让我更加热爱班集体。
```

### CRISPE 提示词输出
## 1. 错别字
| 原词 | 正确写法 | 说明 |
|---|---|---|
| 鼓厉 | 鼓励 | “厉”应为“励”，表激励之意。 |
| 受益非浅 | 受益匪浅 | “非”应为“匪”，意为“不浅”。 |

## 2. 病句与修改
| 原句 | 问题 | 修改建议 |
|---|---|---|
| 同学们也喊着我的名字。 | 表达略显单薄，可增加细节使场景更生动。 | 改为：“同学们也大声喊着我的名字，为我加油鼓劲。” |

## 3. 三维评分
| 维度 | 分数/10 | 理由 |
|---|---:|---|
| 内容 | 8 | 事件完整，有真情实感，但情节较平淡，缺乏细节描写。 |
| 结构 | 8 | 开头、经过、结尾清晰，层次分明，但过渡稍显仓促。 |
| 语言 | 7 | 通顺易懂，有两处错别字，“鼓厉”“非浅”需订正。 |

## 4. 总体评价（50字以内）
选材真实，中心突出，坚持的精神值得肯定。今后多注意字词积累，并丰富细节描写，文章会更生动。

### 糟糕提示词输出
这篇作文写得很用心，结构完整，情感也很真挚，读起来能感受到你在赛场上的坚持和同学们之间的温暖。下面我从几个方面具体说说：

**优点：**
- 叙事清晰，有头有尾——从运动会开始、自己参赛、遇到困难、得到鼓励，到最后坚持完赛和感悟，线索很明确。
- 情感真实，尤其是“听到大家的加油，我仿佛又有了力气”这句，很打动人，写出了集体的力量。
- 主题积极向上，把“坚持比名次更重要”的道理自然地融入其中，不刻意说教。

**可以改进的地方：**
1. **纠正两个错别字**：“鼓厉”应为“鼓励”；“受益非浅”应为“受益匪浅”。
2. **描写可以更生动**：比如“人山人海”可以具体一点，“同学们有的挥舞彩旗，有的喊得脸红脖子粗”；第二圈胸闷腿软时的感受，可以用“像灌了铅一样”“呼吸像被掐住了”这样的比喻，让读者更有画面感。
3. **感悟可以更深入**：除了“坚持比名次重要”，还可以提一句为什么集体的加油声能让人突然有力气，比如“原来一个人的坚持背后，有那么多人在默默支持”，这样主题会更饱满。

**总体评价**：是一篇有真情实感的作文，基础不错，稍加润色就会更出色。如果满分100分，我会给85分。继续加油哦！

## 作文：一次读书分享会（essay_2）

### 原文
上个周末，我和父母一起去图书馆参加读书分享会。走进大厅，整齐的书架和淡淡的书香扑面而来。我选择了一本关于科学家的传记，它讲述了他们面对困难仍然认真研究的故事。通过这次活动，使我懂得了读书不仅能增长知识，还能让人学会思考。回家后，我决定每天坚持阅读二十分钟，并把好词好句记在笔记本上。我很珍惜这次难得的机慧。

### 完整 CRISPE 提示词
```text
你将使用 CRISPE 框架完成作文批改任务。

C - Context（背景）：这是一篇初中生短文，老师需要你做基础语文批改，重点关注错别字、病句、评分与总体评价。
R - Role（角色）：你是认真、客观、表达温和的语文作文批改助手。
I - Input（输入）：作文题目和正文如下。
题目：一次读书分享会
正文：上个周末，我和父母一起去图书馆参加读书分享会。走进大厅，整齐的书架和淡淡的书香扑面而来。我选择了一本关于科学家的传记，它讲述了他们面对困难仍然认真研究的故事。通过这次活动，使我懂得了读书不仅能增长知识，还能让人学会思考。回家后，我决定每天坚持阅读二十分钟，并把好词好句记在笔记本上。我很珍惜这次难得的机慧。
S - Steps（步骤）：
1. 通读全文，只依据作文原文进行判断，不要虚构不存在的问题。
2. 标注作文中的错别字，给出“原词/正确写法/说明”；如果没有错别字，写“未发现明显错别字”。
3. 找出至少 1 处病句或表达不顺的句子，给出修改建议。
4. 从内容、结构、语言三个维度打分，每项满分 10 分，并给出一句理由。
5. 给出 50 字以内的总体评价，语气鼓励但要指出改进方向。
P - Parameters（限制）：回答必须简洁清楚；不要超过 500 字；不要输出与作文无关的内容。
E - Evaluation（输出格式）：请严格按下面 Markdown 格式输出：

## 1. 错别字
| 原词 | 正确写法 | 说明 |
|---|---|---|

## 2. 病句与修改
| 原句 | 问题 | 修改建议 |
|---|---|---|

## 3. 三维评分
| 维度 | 分数/10 | 理由 |
|---|---:|---|
| 内容 |  |  |
| 结构 |  |  |
| 语言 |  |  |

## 4. 总体评价（50字以内）
```

### 糟糕提示词
```text
帮我看看这篇作文写得怎么样：上个周末，我和父母一起去图书馆参加读书分享会。走进大厅，整齐的书架和淡淡的书香扑面而来。我选择了一本关于科学家的传记，它讲述了他们面对困难仍然认真研究的故事。通过这次活动，使我懂得了读书不仅能增长知识，还能让人学会思考。回家后，我决定每天坚持阅读二十分钟，并把好词好句记在笔记本上。我很珍惜这次难得的机慧。
```

### CRISPE 提示词输出
## 1. 错别字
| 原词 | 正确写法 | 说明 |
|---|---|---|
| 机慧 | 机会 | “慧”应为“会”，表示机遇。 |

## 2. 病句与修改
| 原句 | 问题 | 修改建议 |
|---|---|---|
| 通过这次活动，使我懂得了读书不仅能增长知识，还能让人学会思考。 | “通过……使……”导致主语缺失 | 改为“这次活动使我懂得了……”或“通过这次活动，我懂得了……” |

## 3. 三维评分
| 维度 | 分数/10 | 理由 |
|---|---:|---|
| 内容 | 8 | 叙事完整，细节具体，感悟真实。 |
| 结构 | 9 | 时间顺序清晰，开头、经过、结尾层次分明。 |
| 语言 | 7 | 整体通顺，但有错别字和病句，个别表达略显生硬。 |

## 4. 总体评价（50字以内）
文章情感真挚，过程清楚。注意避免“通过…使…”句式，并减少错别字，语言会更流畅。继续加油！

### 糟糕提示词输出
这篇作文整体结构清晰，内容真诚，记录了参加读书分享会的经历和感悟。下面从几个方面帮你分析一下：

**优点：**
1. **主题明确** —— 围绕“读书分享会”展开，从选书、阅读到收获，思路连贯。
2. **情感真实** —— “淡淡的书香扑面而来”这类描写能传递氛围；“珍惜这次机会”表达了对活动的喜爱。
3. **有实际行动** —— 结尾提到“每天坚持阅读”“记好词好句”，说明感悟落到了实处。

**可以改进的地方：**
1. **错别字**：最后一句“机慧”应为“**机会**”。
2. **语句通顺**：文中“通过这次活动，使我懂得了……”有句式杂糅（“通过”和“使”重复使用）。可改为：“**通过这次活动，我懂得了……**”或“**这次活动使我懂得了……**”
3. **内容深度**：可以再具体一些。比如，那本科学家传记中，是哪位科学家？他克服了什么困难？这种细节能让文章更有感染力。
4. **开头与结尾**：开头稍显常规，可尝试用一句感受或场景特写开头；结尾可以升华，比如谈谈读书如何改变了对学习的看法。

**修改示范（仅改动关键部分）：**
> …它讲述了居里夫人在简陋实验室中坚持研究的故事。**通过这次活动，我**懂得了读书不仅能增长知识，还能让人学会思考。回家后，我决定每天坚持阅读二十分钟，并把好词好句记在笔记本上。我很珍惜这次难得的**机会**。

整体来看，这是一篇合格的习作，选材贴近生活，语言朴实真诚。下次写作时注意检查错别字和句子通顺，再适当加入具体事例，会更好！


## 第二部分：RAG 系统构建


In [4]:
rag_result = run_rag_experiment(chat_client, embedding_client, cfg)
display(Markdown(Path(rag_result['markdown_path']).read_text(encoding='utf-8')))



========== 第二部分：RAG 系统构建 ==========
知识库目录：knowledge_base_exp7
文档数量：4
文档块数量：8
Embedding 后端：mock / mock-hash-embedding
向量检索：numpy fallback


RAG 结果已保存：outputs_exp7\exp7_rag_results.md


# 实验七第二部分：RAG 系统构建结果

生成时间：2026-06-04 11:14:36
LLM 后端：deepseek / deepseek-v4-flash
Embedding 后端：mock / mock-hash-embedding
向量数据库：numpy fallback

## 知识库文档列表及内容摘要

| 文档 | 摘要 |
|---|---|
| company_leave_policy.txt | 公司请假制度

第一条 适用范围：本制度适用于公司全体正式员工、试用期员工以及经部门负责人确认的实习人员。员工因病、因事、婚丧、生育、学习考试等原因不能按时出勤时，应按照本制度办理... |
| office_equipment_policy.txt | 办公设备申领与归还流程

第一条 设备范围：本流程所称办公设备包括笔记本电脑、台式机、显示器、键盘鼠标、耳机、移动硬盘、投影仪、会议摄像头以及经行政部登记的其他办公资产。设备由行政... |
| overtime_allowance_policy.txt | 公司加班与补贴政策

第一条 加班定义：加班是指员工因工作需要，在标准工作时间之外继续完成经批准的工作任务。员工自愿延长工作时间但未经过审批的，不计为公司认可的加班。所有加班应坚持... |
| remote_work_policy.txt | 远程办公与信息安全规定

第一条 申请条件：远程办公适用于因项目协作、出差、特殊天气、临时照护家庭成员或其他经公司认可的情形。员工申请远程办公时，应明确远程日期、工作地点、主要任务... |

## 分块统计

共加载 4 个文档，切分为 8 个文档块。

## 测试问题及模型回答记录表

| 类型 | 问题 | 模型回答 | 来源引用 |
|---|---|---|---|
| 直接问题 | 病假需要在什么时候提交申请，返岗后需要补交什么材料？ | 回答：病假需在上班前通过企业微信提交申请，返岗后应在三个工作日内补交医院诊断证明、病历或正规医疗机构开具的休息建议。<br><br>依据：company_leave_policy.txt | company_leave_policy.txt(company_leave_policy_chunk_001, score=0.4307)；company_leave_policy.txt(company_leave_policy_chunk_000, score=0.3905)；office_equipment_policy.txt(office_equipment_policy_chunk_000, score=0.3543)；overtime_allowance_policy.txt(overtime_allowance_policy_chunk_000, score=0.3445) |
| 直接问题 | 工作日晚间加班满两小时有什么补贴？超过四小时并晚于二十二点结束怎么处理？ | 回答：工作日晚间加班满两小时，给予三十元餐补。若超过四小时且结束时间晚于二十二点，可报销单程交通费。<br><br>依据：overtime_allowance_policy.txt | overtime_allowance_policy.txt(overtime_allowance_policy_chunk_000, score=0.5963)；remote_work_policy.txt(remote_work_policy_chunk_000, score=0.4563)；overtime_allowance_policy.txt(overtime_allowance_policy_chunk_001, score=0.4408)；company_leave_policy.txt(company_leave_policy_chunk_001, score=0.4239) |
| 综合问题 | 员工周末加班后想在下周调休，应该怎么申请，调休和加班记录之间有什么关系？ | 回答：员工周末加班后如需在下周调休，应在请假系统中选择“调休假”类型，并关联已审批的加班单来完成申请。调休必须基于已审批的加班记录，且周末加班形成的调休应在三个月内使用。<br><br>依据：overtime_allowance_policy.txt；company_leave_policy.txt | overtime_allowance_policy.txt(overtime_allowance_policy_chunk_000, score=0.5561)；overtime_allowance_policy.txt(overtime_allowance_policy_chunk_001, score=0.5327)；company_leave_policy.txt(company_leave_policy_chunk_001, score=0.4953)；remote_work_policy.txt(remote_work_policy_chunk_000, score=0.4191) |
| 综合问题 | 新员工需要领电脑并偶尔远程办公时，设备申领和信息安全方面分别要注意什么？ | 回答：新员工申领电脑时，须由直属主管提前三个工作日提交设备申领单，行政部按标准配置电脑和外设，信息技术部完成系统初始化、杀毒软件安装、磁盘加密和公司邮箱配置，员工领取时需核对资产编号并签署《办公资产领用确认单》。对于远程办公，原则上应使用公司发放的电脑，必须连接公司VPN或经批准的安全访问通道，不得通过公共网盘或个人邮箱等未授权方式传输公司文件；处理重要数据时要开启磁盘加密和屏幕锁定，离开座位时及时锁屏。<br><br>依据：[来源1] remote_work_policy.txt；[来源2] office_equipment_policy.txt | remote_work_policy.txt(remote_work_policy_chunk_000, score=0.5634)；office_equipment_policy.txt(office_equipment_policy_chunk_000, score=0.5338)；remote_work_policy.txt(remote_work_policy_chunk_001, score=0.4819)；overtime_allowance_policy.txt(overtime_allowance_policy_chunk_000, score=0.4346) |
| 知识库外 | 公司的股票期权什么时候兑现？ | 知识库中未找到与该问题直接相关的政策依据，因此无法回答。建议咨询人力资源部或查看补充制度文件。 | office_equipment_policy.txt(office_equipment_policy_chunk_000, score=0.3114)；remote_work_policy.txt(remote_work_policy_chunk_001, score=0.3113)；remote_work_policy.txt(remote_work_policy_chunk_000, score=0.3032)；overtime_allowance_policy.txt(overtime_allowance_policy_chunk_001, score=0.2982) |

## 简要分析

直接问题通常能命中单个制度片段，回答较稳定；综合问题需要把请假、加班、设备、远程办公等片段合并，来源引用能帮助核查；知识库外问题应拒答，避免模型凭常识编造公司制度。

## 核心流程说明

1. 读取 TXT/PDF 文档。
2. 按固定长度与重叠窗口切分文档块。
3. 通过 EmbeddingClient 生成向量。
4. 使用 FAISS IndexFlatIP 存储并按相似度检索。
5. 将检索片段作为上下文交给 LLM 生成答案并保留来源。

## 第三部分：简易 Agent 设计


In [5]:
agent_result = run_agent_experiment('人工智能在医疗领域的应用', cfg.output_dir)
display(Markdown((cfg.output_dir / 'exp7_agent_trace.md').read_text(encoding='utf-8')))



========== 第三部分：简易 Agent 设计 ==========
Agent 记录已保存：outputs_exp7\exp7_agent_trace.md


# 实验七第三部分：简易 Agent 设计结果

主题：人工智能在医疗领域的应用

## Agent 工作流程步骤描述

用户任务 → 规划搜索方向 → 调用搜索工具 → 观察搜索结果 → 调用摘要工具 → 判断是否满足报告要求 → 生成 Markdown 报告

## 工具定义代码/伪代码

```python
def search_tool(query: str, top_k: int = 4) -> list[dict]:
    return simulated_search_results

def summarize_tool(search_results: list[dict]) -> dict:
    return {'overview': ..., 'findings': ..., 'sources': ...}
```

## 思考-行动-观察记录

| 轮次 | 思考 | 行动 | 观察 |
|---:|---|---|---|
| 1 | 先明确主题并收集基础资料，优先获取应用场景、价值和风险信息。 | `search_tool({'query': '人工智能在医疗领域的应用', 'top_k': 4})` | 搜索工具返回 4 条结果，覆盖影像诊断、临床决策、药物研发和隐私安全。 |
| 2 | 搜索结果已经覆盖多个角度，下一步调用摘要工具提炼关键发现。 | `summarize_tool({'result_count': 4})` | 摘要工具生成了主题概述、3个关键发现、风险提示和信息来源列表。 |
| 3 | 信息已足够生成结构化报告，检查报告是否包含主题概述、3个关键发现和来源。 | `generate_agent_report(summary)` | 报告已生成，包含主题概述、关键发现、风险提示和模拟来源，任务完成。 |

## 最终生成报告

# 信息收集与整理 Agent 报告：人工智能在医疗领域的应用

## 主题概述
人工智能在医疗领域主要承担辅助分析、信息整合和效率提升的角色，不能替代医生独立诊断。

## 3个关键发现
1. 医学影像是医疗 AI 最常见的落地场景，可辅助医生进行筛查和提高效率。
2. 临床决策支持可以把病历、检验指标和指南结合起来，为医生提供风险提醒。
3. 药物研发是 AI 的重要方向，但候选结果仍需实验和临床验证。

## 风险提示
- 医疗 AI 必须重视隐私保护、数据安全、模型偏差和责任边界。

## 信息来源
- 模拟来源A：医疗AI行业综述
- 模拟来源B：医院信息化白皮书
- 模拟来源C：AI 制药案例集
- 模拟来源D：医疗数据治理指南

## 输出文件检查


In [6]:
for path in sorted(cfg.output_dir.glob('*')):
    if path.is_file():
        print(path)


outputs_exp7\exp7_agent_report.md
outputs_exp7\exp7_agent_trace.json
outputs_exp7\exp7_agent_trace.md
outputs_exp7\exp7_prompt_engineering_results.json
outputs_exp7\exp7_prompt_engineering_results.md
outputs_exp7\exp7_rag_chunks.json
outputs_exp7\exp7_rag_results.json
outputs_exp7\exp7_rag_results.md
outputs_exp7\exp7_run_summary.json
outputs_exp7\exp7_run_summary.txt
outputs_exp7\notebook_execute_log.txt
outputs_exp7\run_log_deepseek.txt
